# 使用 W&B Weave 监控 OpenAI API

<img src="http://wandb.me/logo-im-png" width="400" alt="Weights & Biases" />

<!--- @wandbcode{weave_openai_client_qs} -->

<a target="_blank" href="https://colab.research.google.com/github/wandb/weave/blob/master/examples/prompts/llm_monitoring/openai_client_quickstart.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**注意：** 运行此 colab 需要一个 [OpenAI API 密钥](https://platform.openai.com/account/api-keys)。

使用 W&B 的 OpenAI 集成来监控 OpenAI API 调用，了解你的项目和团队如何使用 LLM。
在本示例中，我们将生成模板化的 Weave Board：一种可以在 UI 中浏览和自定义的 LLM 使用情况监控仪表盘。

* 自动跟踪 LLM 使用情况，并在各项目/团队之间汇总成本、延迟和吞吐量等有用指标
* 动态查询所有 OpenAI API 调用的日志并从中获取洞察
* 以可视化方式迭代地切分、聚合和探索数据；自定义面板以聚焦有趣的模式；通过交互式仪表盘更轻松地与团队分享进展

<img src="https://raw.githubusercontent.com/wandb/weave/master/docs/assets/full_board_view.png">

[体验此 Weave Board 的在线版本 →](http://wandb.me/llm-monitoring-board)

#### 初次使用 Weights & Biases？[-> 在这里注册账户 <-](https://wandb.ai/site)

# 第 0 步：准备工作

安装依赖，登录 W&B 以便保存和分享你的工作，并完成 OpenAI 身份验证。

In [ ]:
# if not already installed
!pip install -qqq weave openai tiktoken wandb

In [ ]:
import wandb
wandb.login()

In [ ]:
import weave
import os
WANDB_BASE_URL = "https://api.wandb.ai"
os.environ["WANDB_BASE_URL"] = WANDB_BASE_URL

In [ ]:
# authenticate with OpenAI
from getpass import getpass

if os.getenv("OPENAI_API_KEY") is None:
  os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI key from: https://platform.openai.com/account/api-keys\n")
assert os.getenv("OPENAI_API_KEY", "").startswith("sk-"), "This doesn't look like a valid OpenAI API key"
print("OpenAI API key configured")

# 第 1 步：在 W&B 中配置数据流和存储

将 WB_ENTITY 设置为你的 wandb 用户名或团队名。登录 W&B 并前往主页 [wandb.ai/home](https://wandb.ai/home)，在左侧边栏的“Profile”和“Teams”下查看可用选项。

In [ ]:
WB_ENTITY = "" # set to your wandb username or team name
WB_PROJECT = "weave" # top-level directory for this work
STREAM_NAME = "openai_logs" # record table which stores the logs of OpenAI API calls as they stream in

# 第 2 步：调用 init_monitor()

要开始监控 OpenAI API 的使用情况，请调用 `init_monitor(<stream>)`，其中 `<stream>` 的形式为 `<wandb_team_or_user>/<wandb_project>/<stream_name>`。该数据流会记录并存储所有 OpenAI API 调用。

运行此单元格会打印一个链接，可在 Weave UI 中查看当前项目。

In [ ]:
from weave.monitoring import openai, init_monitor
m = init_monitor(f"{WB_ENTITY}/{WB_PROJECT}/{STREAM_NAME}")

# specifying a single model for simplicity
OPENAI_MODEL = 'gpt-3.5-turbo'

# prefill with some sample logs
r = openai.ChatCompletion.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": "hello world!"}])
r = openai.ChatCompletion.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": "what is 2+2?"}])

# 第 3 步：预览监控仪表盘

点击上面的链接预览数据流，然后点击右侧边栏中的“OpenAI Monitor Board”，为该数据流创建一个 Weave Board。

<img src="https://raw.githubusercontent.com/wandb/weave/master/docs/assets/short_board_attempt.gif" width=75%>

# 第 4 步：探索并理解你的 LLM 使用情况

要保存你的工作，请点击页面顶部自动生成的名称来重命名该 Board。要分享你的 Board，请点击右上角的 \"Publish\"。

<img src="https://raw.githubusercontent.com/wandb/weave/master/docs/assets/publish_board_short.gif" width=75%>

要在迭代过程中实时可视化你的工作，你可以：
* 在单独的标签页中保持 Board 打开，刷新即可查看最新数据
* 随时重命名 Board 以便查找，并 \"Publish\" 该版本以便与他人分享链接
* 在 weave.wandb.ai 中进入相应的 W&B entity 和 W&B 项目名称，找到之前保存的 Board
* 或者打开一个新的 Board 模板实例，基于目前累积的所有数据重新开始


接下来，我们将演示几种跟踪 OpenAI API 调用的方式。根据你的用例，还有更多可能性，我们非常期待看到你基于这些入门模板创造出的作品。

# 示例

## 示例 0：记录提示词及其补全

监控一个 ChatCompletion 请求并打印相应的响应，只提取补全的文本。

In [ ]:
response = openai.ChatCompletion.create(model=OPENAI_MODEL, messages=[
        {"role": "user", "content": f"What is the meaning of life, the universe, and everything?"},
    ])
print(response['choices'][0]['message']['content'])

## 示例 1：将相关参数作为属性进行跟踪

提取感兴趣的参数，并将其作为属性记录在日志记录中。
这里我们将 "system prompt" 与 "prompt template" 以及 "equation" 参数分开跟踪。这一次我们将打印 ChatCompletion 调用返回的完整结构化响应。

In [ ]:
system_prompt = "you always write in bullet points"
prompt_template = 'solve the following equation step by step: {equation}'
params = {'equation': '4 * (3 - 1)'}
openai.ChatCompletion.create(model=OPENAI_MODEL,
                             messages=[
                                    {"role": "system", "content": system_prompt},
                                    {"role": "user", "content": prompt_template.format(**params)},
                                ],
                             # you can add additional attributes to the logged record
                             # see the monitor_api notebook for more examples
                             monitor_attributes={
                                 'system_prompt': system_prompt,
                                 'prompt_template': prompt_template,
                                 'params': params
                             })

## 示例 2：记录持续的消息流

监控一个消息流，并将结果记录为单条记录。注意：这种格式下不会统计 token。

In [ ]:
from weave.monitoring.openai import message_from_stream
r = openai.ChatCompletion.create(model=OPENAI_MODEL, messages=[
        {"role": "system", "content": "You are a robot and only speak in robot, like beep bloop bop."},
        {"role": "user", "content": "Tell me a 50-word story."},
    ], stream=True)
for s in message_from_stream(r):
    print(s, end='')

## 示例 3：组织提示词工程实验

这里我们比较了 system prompt、用户问题和目标受众的几种简单选项。不妨尝试你自己的实验，在 Board 中按不同参数分组探索，看看能否发现有趣的洞察。

In [ ]:
def explain_math(system_prompt, prompt_template, params):
    openai.ChatCompletion.create(model=OPENAI_MODEL,
                             messages=[
                                    {"role": "system", "content": system_prompt},
                                    {"role": "user", "content": prompt_template.format(**params)},
                                ],
                             # you can add additional attributes to the logged record
                             # see the monitor_api notebook for more examples
                             monitor_attributes={
                                 'system_prompt': system_prompt,
                                 'prompt_template': prompt_template,
                                 'params': params
                             })

In [ ]:
# feel free to substitute your own prompts :)
system_prompts = ["you're extremely flowery and poetic", "you're very direct and precise", "balance brevity with insight"]
prompt_template = 'explain the solution of the following to a {audience}: {equation}'
equations = ['x^2 + 4x + 9 = 0', '15 * (2 - 6) / 4']
audience = ["new student", "math genius"]

for system_prompt in system_prompts:
    for equation in equations:
        for person in audience:
            params = {"equation" : equation, "audience" : person}
            explain_math(system_prompt, prompt_template, params)